### Give the AI a constantly updated "brain" of crypto news and teach it how to search before answering.

## Step 1: Data Collection & Raw Storage (MongoDB)

Begin by scraping crypto news articles from theblock.co and storing them in MongoDB.

At this stage:
- Data is stored as-is (raw format)
- Includes HTML content, metadata, and structure
- Acts as a permanent source of truth

## Step 2: Preprocessing & CSV Export (Optimization Layer)

Once the raw data is collected, Introduce an intermediate step to make downstream processing more efficient.
Why this step is needed

Raw MongoDB data has a few issues:
- Contains HTML tags (not clean text)
- Requires repeated database queries during processing
- Slows down experimentation and iteration

So instead of directly processing from MongoDB every time, we:

> Extract → Clean → Store in CSV

In [ ]:
!pip install langchain-chroma langchain-huggingface langchain-openai langchain-text-splitters pymongo sentence-transformers langchain-community rank_bm25 langchain-groq

In [3]:
import re
import os
import csv
import pandas as pd
import html as html_lib
from bs4 import BeautifulSoup
from pymongo import MongoClient
from dotenv import load_dotenv

load_dotenv()

False

In [ ]:
MONGO_URI = os.getenv("MONGO_URI","")
DATABASE_NAME = "rag_pipeline"
COLLECTION_NAME = "articles_raw"
OUTPUT_CSV = "output.csv"
BATCH_SIZE = 1000
MIN_PARAGRAPH_LENGTH = 30

In [5]:
client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)

client.admin.command("ping")
print("Connected ✓")

db = client[DATABASE_NAME]
collection = db[COLLECTION_NAME]

total = collection.count_documents({})

Connected ✓


In [6]:
def extract_paragraphs(raw_html: str) -> list[str]:
    """
    Parse raw HTML and extract clean text from every <p> tag.
    Each paragraph is independently cleaned so it can later be
    chunked / embedded individually if needed.
    """
    soup = BeautifulSoup(raw_html, "html.parser")

    paragraphs = []
    boilerplate_text = "The following article is adapted from The Block’s newsletter, The Daily , which comes out on weekday afternoons."
    # Compile a regex that handles variable whitespace within the boilerplate text
    boilerplate_regex = re.compile(re.escape(boilerplate_text).replace("\\ ", "\\s*"), re.IGNORECASE)

    for p_tag in soup.find_all("p"):
        # Get inner text, collapsing nested tags (<strong>, <a>, <em>, etc.)
        text = p_tag.get_text(separator=" ")

        # Decode any remaining HTML entities (&amp; → &, &nbsp; → space …)
        text = html_lib.unescape(text)

        # Remove the boilerplate text
        text = boilerplate_regex.sub("", text)

        # Collapse whitespace
        text = re.sub(r"\s+", " ", text).strip()

        # Remove non-printable / control characters
        text = re.sub(r"[^\x20-\x7E\u00A0-\uFFFF]", "", text)

        # Drop boilerplate / noise paragraphs that are too short
        if len(text) >= MIN_PARAGRAPH_LENGTH:
            paragraphs.append(text)

    return paragraphs


def preprocess_content(raw_html: str) -> str:
    """
    Returns a single clean string of all meaningful paragraphs,
    separated by double newlines — ideal for RAG ingestion.

    Downstream options:
      • Feed the full string to a text splitter (LangChain, LlamaIndex, etc.)
      • Or split on '\n\n' to get individual paragraph chunks right away.
    """
    if not isinstance(raw_html, str):
        raw_html = str(raw_html) if raw_html is not None else ""

    paragraphs = extract_paragraphs(raw_html)

    # Join with blank-line separator — preserves semantic paragraph boundaries
    return "\n\n".join(paragraphs)

In [7]:
cursor = collection.find({}, batch_size=BATCH_SIZE)

processed = 0
skipped = 0
empty_after = 0   # had content but no usable <p> text

with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as csvfile:
    writer = None  # initialised after reading the first document

    for doc in cursor:
        if "content" in doc:
            cleaned = preprocess_content(doc["content"])
            if not cleaned:
                empty_after += 1
            doc["content"] = cleaned
        else:
            skipped += 1

        if writer is None:
            fieldnames = list(doc.keys())
            writer = csv.DictWriter(
                csvfile,
                fieldnames=fieldnames,
                extrasaction="ignore",
                quoting=csv.QUOTE_ALL,
            )
            writer.writeheader()

        writer.writerow(doc)
        processed += 1

        if processed % 2000 == 0:
            print(f"  Processed {processed:,} / {total:,} …")

print(f"\nDone! {processed:,} documents written to '{OUTPUT_CSV}'.")
if skipped:
    print(f"  ⚠  {skipped} documents had no 'content' field.")
if empty_after:
    print(f"  ⚠  {empty_after} documents had no usable <p> text after cleaning.")

client.close()

  Processed 2,000 / 22,640 …
  Processed 4,000 / 22,640 …
  Processed 6,000 / 22,640 …
  Processed 8,000 / 22,640 …
  Processed 10,000 / 22,640 …
  Processed 12,000 / 22,640 …
  Processed 14,000 / 22,640 …
  Processed 16,000 / 22,640 …
  Processed 18,000 / 22,640 …
  Processed 20,000 / 22,640 …
  Processed 22,000 / 22,640 …

Done! 22,640 documents written to 'output.csv'.
  ⚠  15 documents had no usable <p> text after cleaning.


In [8]:
df = pd.read_csv("/content/output.csv")
df

,_id,article_id,content,published_at,slug,source,tags,thumbnail,title,url,type
0,69d3d49857d387f7e016c712,396391,Below is a summarized version of The Block Res...,2026-04-06T10:10:58-04:00,a-deep-dive-into-the-future-of-onchain-liquidi...,theblock,"['1inch', '1inch-exchange', '1inch-network', '...",https://www.tbstat.com/wp/uploads/2026/04/2025...,A Deep Dive into the Future of Onchain Liquidi...,https://www.theblock.co/post/396391/a-deep-div...,news
1,69d3d64a57d387f7e016c71c,395685,Below is a summarized version of The Block Res...,2026-03-30T10:11:34-04:00,strategic-selection-a-practical-guide-to-choos...,theblock,[],https://www.tbstat.com/wp/uploads/2026/03/2025...,Strategic Selection: A Practical Guide to Choo...,https://www.theblock.co/post/395685/strategic-...,news
2,69d3d64c57d387f7e016c71d,386700,Perpetual decentralized exchanges (perpDEXs) p...,2026-01-22T10:02:39-05:00,edgex-where-mobile-traders-meet-institutional-...,theblock,"['dex', 'edgex']",https://www.tbstat.com/wp/uploads/2026/01/2026...,edgeX: Where Mobile Traders Meet Institutional...,https://www.theblock.co/post/386700/edgex-wher...,news
3,69d3d64d57d387f7e016c71e,383020,Welcome to The Block’s 2026 Digital Assets Out...,2025-12-17T11:57:49-05:00,2026-digital-assets-outlook-report,theblock,['stablecoin'],https://www.tbstat.com/wp/uploads/2025/12/2025...,2026 Digital Assets Outlook Report,https://www.theblock.co/post/383020/2026-digit...,news
4,69d3d64f57d387f7e016c71f,373848,The DeFi landscape remains highly fragmented. ...,2025-10-08T11:58:17-04:00,yellow-a-clearing-network-unifying-fragmented-...,theblock,"['scaling', 'yellow', 'yellow-network']",https://www.tbstat.com/wp/uploads/2025/10/2025...,Yellow: A Clearing Network Unifying Fragmented...,https://www.theblock.co/post/373848/yellow-a-c...,news
...,...,...,...,...,...,...,...,...,...,...,...
22635,69d88da457d387f7e017fe3b,396847,Blockchain investigator ZachXBT has uncovered ...,2026-04-09T07:55:02-04:00,zachxbt-uncovers-north-korea-linked-it-worker-...,theblock,"['dprk', 'hacks', 'identity', 'north-korea-cry...",https://www.tbstat.com/wp/uploads/2021/12/2021...,ZachXBT uncovers North Korea-linked IT worker ...,https://www.theblock.co/post/396847/zachxbt-un...,news
22636,69d88da557d387f7e017fe3c,396848,A solo bitcoin miner beat monumental odds to s...,2026-04-09T07:44:59-04:00,lucky-solo-bitcoin-miner-beats-roughly-one-in-...,theblock,"['bitcoin', 'mining-companies']",https://www.tbstat.com/wp/uploads/2023/12/2023...,Lucky solo bitcoin miner beats roughly one-in-...,https://www.theblock.co/post/396848/lucky-solo...,news
22637,69d88da657d387f7e017fe3d,396836,"Bitcoin may have already found its floor, acco...",2026-04-09T06:19:26-04:00,strategys-michael-saylor-says-bitcoin-likely-b...,theblock,"['analyst-reports', 'crypto-movers', 'michael-...",https://www.tbstat.com/wp/uploads/2024/11/2024...,Strategy's Michael Saylor says bitcoin likely ...,https://www.theblock.co/post/396836/strategys-...,news
22638,69d88da657d387f7e017fe3e,396833,The U.S. Department of Justice and the Commodi...,2026-04-09T06:08:02-04:00,doj-cftc-argue-sports-event-contracts-are-fina...,theblock,"['arizona', 'cftc', 'court-hearings', 'doj', '...",https://www.tbstat.com/wp/uploads/2023/04/2023...,"DOJ, CFTC argue Kalshi's sports and event cont...",https://www.theblock.co/post/396833/doj-cftc-a...,news


In [9]:
df1 = df[df.published_at.isnull()]
missing_published_at = df1.article_id.unique()
print(f"Found {len(missing_published_at)} unique article IDs with null 'published_at' values.")

Found 0 unique article IDs with null 'published_at' values.


In [10]:
df1 = df.copy()
# The offset -04:00 is already in the string, so just parse + convert
df1['published_at'] = (
    pd.to_datetime(df1['published_at'], format='ISO8601', utc=True)
    .dt.tz_convert('UTC')
)
df1.sort_values(by='published_at', ascending=False, inplace=True)
df1['published_at'] = df1['published_at'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
df1.head()

,_id,article_id,content,published_at,slug,source,tags,thumbnail,title,url,type
22620,69d88d9657d387f7e017fe2c,396959,"Covenant AI, a major subnet developer on Bitte...",2026-04-10T02:42:49Z,covenant-ai-exits-bittensor-tao,theblock,['ai'],https://www.tbstat.com/wp/uploads/2025/07/bitt...,'It is decentralization theatre': Covenant AI ...,https://www.theblock.co/post/396959/covenant-a...,news
22621,69d88d9757d387f7e017fe2d,396948,Galaxy's stock is up over 11% on the day follo...,2026-04-09T20:46:28Z,galaxy-stock-rallies-11-after-annual-report-sh...,theblock,"['crypto-banks', 'fintech', 'staking-firms']",https://www.tbstat.com/wp/uploads/2023/03/2023...,Galaxy stock rallies 11% after annual report s...,https://www.theblock.co/post/396948/galaxy-sto...,news
22622,69d88d9857d387f7e017fe2e,396943,Bitcoin (BTC) and ether (ETH) prices have move...,2026-04-09T20:11:49Z,cryptoquant-bitcoin-ether-price-rally-perpetua...,theblock,"['bitcoin', 'ethereum', 'tokens']",https://www.tbstat.com/wp/uploads/2023/03/2023...,"CryptoQuant says bitcoin, ether rally driven b...",https://www.theblock.co/post/396943/cryptoquan...,news
22623,69d88d9957d387f7e017fe2f,396926,"In his more than 300-page memoir, Binance foun...",2026-04-09T19:54:38Z,binance-changpeng-zhao-claims-crypto-rivals-pa...,theblock,"['changpeng-zhao', 'cz']",https://www.tbstat.com/wp/uploads/2023/04/2023...,Binance's Changpeng Zhao claims U.S. crypto ri...,https://www.theblock.co/post/396926/binance-ch...,news
22624,69d88d9a57d387f7e017fe30,396922,Tornado Cash developer Roman Storm and the Dep...,2026-04-09T18:43:06Z,tornado-cash-developer-roman-storms-fate-uncle...,theblock,"['crime', 'roman-storm', 'tornado-cash']",https://www.tbstat.com/wp/uploads/2025/07/torn...,"'This is a lot,' judge says as Tornado Cash de...",https://www.theblock.co/post/396922/tornado-ca...,news


## Step 3: Data Processing (Chunking + Embedding)
Raw articles are too large for efficient search, so we process them.


In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.documents import Document
from langchain_chroma import Chroma

from typing import Any

In [12]:
documents = list(df1.to_dict(orient="records"))
documents_list = list(documents[:1000])

In [13]:
documents_list[0]

{'_id': '69d88d9657d387f7e017fe2c',
 'article_id': 396959,
 'content': 'Covenant AI, a major subnet developer on Bittensor , announced Thursday that it is leaving the AI-focused network ecosystem.\n\nIn a statement on X, Covenant AI Founder Sam Dare said the company can no longer build on Bittensor because the network\'s governance contradicts its stated commitment to decentralization.\n\n"The entire premise of Bittensor, the promise that drew builders, miners, validators, and investors into this ecosystem, is that no single entity controls it," Dare wrote. "That promise is a lie."\n\nThe Covenant founder said the team has been building on a conviction that AI model training should not be controlled by a single entity, and delivered on that mission by developing Covenant-72B into the largest decentralized case of LLM pre-training run in history. Such efforts placed Covenant as one of the most prominent subnets on Bittensor.\n\nDespite the success, Covenant has decided to leave the netw

In [14]:
def build_full_text(doc: dict[str, Any]) -> str:
    title = (doc.get("title") or "").strip()
    content = doc.get("content")
    tags = ", ".join(tag.strip().lower() for tag in doc.get("tags", []) if isinstance(tag, str))

    parts = []
    if title:
        parts.append(f"Title: {title}")
    if tags:
        parts.append(f"Tags: {tags}")
    if content:
        parts.append(f"Content: {content}")

    return "\n\n".join(parts)

def build_splitter(chunk_size: int, chunk_overlap: int) -> RecursiveCharacterTextSplitter:
    return RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )


def build_metadata(doc: dict[str, Any], chunk_index: int) -> dict[str, Any]:
    tags = [tag.strip().lower() for tag in doc.get("tags", []) if isinstance(tag, str)]
    return {
        "article_id": str(doc.get("article_id") or ""),
        "title": (doc.get("title") or "")[:500],
        "url": doc.get("url") or "",
        "slug": doc.get("slug") or "",
        "tags": ", ".join(tags),
        "published_at": doc.get("published_at"),
        "chunk_index": chunk_index,
    }

def split_document(doc: dict[str, Any], splitter: RecursiveCharacterTextSplitter) -> tuple[str | None, list[str]]:
    article_id = doc.get("article_id")
    if article_id is None:
        print("Skipping document without article_id: mongo_id=%s", doc.get("_id"))
        return None, []

    full_text = build_full_text(doc)
    if not full_text:
        print("Skipping empty document for article_id=%s", article_id)
        return None, []

    return str(article_id), splitter.split_text(full_text)


def chunk_document(doc: dict[str, Any], splitter) -> list[dict[str, Any]]:
    article_id, chunks = split_document(doc, splitter)
    if article_id is None:
        return []

    records: list[dict[str, Any]] = []
    for chunk_index, chunk in enumerate(chunks):
        records.append(
            {
                "id": f"{article_id}_{chunk_index}",
                "text": chunk,
                "metadata": build_metadata(doc, chunk_index),
            }
        )

    return records

def chunk_documents(docs: list[dict[str, Any]], splitter) -> list[dict[str, Any]]:
    if not docs:
        return []

    records: list[dict[str, Any]] = []

    for doc in docs:
        doc_records = chunk_document(doc, splitter)
        records.extend(doc_records)

    return records

In [15]:
# Chunk the corpus so retrieval is focused and fits the LLM context
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

In [ ]:
# HuggingFace embeddings for dense retrieval (runs locally, no extra API cost)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

In [17]:
# Dense retriever: Chroma vector store + embedding model
vector_store = Chroma(
        collection_name='articles',
        persist_directory='data/chrome',
        embedding_function=embeddings,
)

In [ ]:
def add_records_to_chroma(vectorstore: Chroma, records: list[dict[str, Any]]) -> int:
    if not records:
        return 0

    ids = [record["id"] for record in records]
    texts = [record["text"] for record in records]
    metadatas = [record["metadata"] for record in records]
    vectorstore.add_texts(texts=texts, metadatas=metadatas, ids=ids)
    return len(records)


records = chunk_documents(documents_list, splitter)
print('No. of chuncks to process: ',len(records))
batch_size = 100

persisted_chunks = 0
for start in range(0, len(records), batch_size):
    persisted_chunks += add_records_to_chroma(
        vector_store,
        records[start:start + batch_size],
    )
    print(f"Processed {persisted_chunks} chunks...")

In [19]:
# dense_retriever = vector_store.as_retriever(search_kwargs={"k": 4})
dense_retriever = vector_store.as_retriever()
print("Dense retriever ready (semantic search)")

Dense retriever ready (semantic search)


In [20]:
# Sparse retriever: BM25 ranks by keyword overlap
# Convert dictionaries to LangChain Document objects
lc_documents = []
for doc_dict in documents_list:
    page_content = doc_dict.get('content', '')
    metadata = {k: v for k, v in doc_dict.items() if k != 'content'}
    lc_documents.append(Document(page_content=page_content, metadata=metadata))

sparse_retriever = BM25Retriever.from_documents(lc_documents)
# sparse_retriever.k = 4
print("Sparse retriever ready (keyword search)")

Sparse retriever ready (keyword search)


## Step 4: Hybrid Search (Best of Both Worlds)

When a user asks a question, don't rely on just one search method.

In [21]:
# The Hybrid Approach: Combining Both Worlds
# Mathematical Formulation
# For each document, you compute two separate scores and combine them:
# final_score = (alpha × sparse_score) + ((1 - alpha) × dense_score)
# Hybrid retriever combines both signals
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[0.3, 0.7],
)
print("Hybrid retriever assembled (dense + sparse)")

Hybrid retriever assembled (dense + sparse)


In [22]:
# Quick probe to see what hybrid returns
query = "What's happening with Bitcoin ETFs?"
results = hybrid_retriever.invoke(query)

print(f"Query: {query}")
for i, doc in enumerate(results, 1):
    print(f"{i}. {doc.page_content[:180]}...")

Query: What's happening with Bitcoin ETFs?
1. Happy Thursday! Bitcoin (BTC) is holding a tight range around $68,000 as macro forces squeeze liquidity, with analysts saying the market is in a compression phase awaiting fresh de...
2. Happy Wednesday! Bitcoin pushed back above $73,000 this morning as spot ETF inflows pick up again, with analysts saying the market has so far weathered Middle East tensions better ...
3. Happy Monday! Bitcoin (BTC) is trading back above $69,000 despite escalating Middle East tensions and fading rate-cut hopes, with analysts citing contained liquidations, short-live...
4. Happy Wednesday! Bitcoin (BTC) rebounded to $69,000 this morning after briefly touching its year-to-date low yesterday, with Wintermute analysts attributing the move largely to sho...
5. The move follows $171 million in net outflows from U.S. spot bitcoin ETFs on March 26 , alongside continued outflows from ether funds, which together extend a broader pattern of un...
6. Meanwhile, institut

In [23]:
# Compare dense vs sparse vs hybrid for the same query
test_query = "What's happening with Bitcoin ETFs?"

dense_only = dense_retriever.invoke(test_query)
sparse_only = sparse_retriever.invoke(test_query)
hybrid = hybrid_retriever.invoke(test_query)

print(f"Query: {test_query}\n")
print("Dense:")
for i, doc in enumerate(dense_only, 1):
    print(f" {i}. {doc.page_content[:120]}...")

print("\nSparse:")
for i, doc in enumerate(sparse_only, 1):
    print(f" {i}. {doc.page_content[:120]}...")

print("\nHybrid:")
for i, doc in enumerate(hybrid, 1):
    print(f" {i}. {doc.page_content[:120]}...")

Query: What's happening with Bitcoin ETFs?

Dense:
 1. The move follows $171 million in net outflows from U.S. spot bitcoin ETFs on March 26 , alongside continued outflows fro...
 2. Meanwhile, institutional capital recently retreated for the first time in four weeks.

Bitcoin spot ETFs recorded $296 m...
 3. Ether’s ETF complex has been steadier but less decisive. U.S.-listed spot ether ETFs saw about $38.7 million of net infl...
 4. This trend is also spotted in the continued outflows from U.S. crypto exchange-traded funds. Bitcoin ETFs recently poste...

Sparse:
 1. Happy Thursday! Bitcoin (BTC) is holding a tight range around $68,000 as macro forces squeeze liquidity, with analysts s...
 2. Happy Wednesday! Bitcoin pushed back above $73,000 this morning as spot ETF inflows pick up again, with analysts saying ...
 3. Happy Monday! Bitcoin (BTC) is trading back above $69,000 despite escalating Middle East tensions and fading rate-cut ho...
 4. Happy Wednesday! Bitcoin (BTC) rebounded

## Step 5: Cross-Encoder Reranking (Refining Results)
Problem: Hybrid retrieval gives us candidates, but they're not perfectly ordered. A document ranked #8 might actually be more relevant than #1.

Solution: Cross-encoders process [query, document] pairs jointly and output a relevance score. Unlike bi-encoders (used in dense retrieval), they see both inputs together, enabling deeper understanding.

Trade-off:
- Much slower than bi-encoders (can't pre-compute)
- Much more accurate (full attention between query and doc)

Production Pattern: Retrieve 50-100 candidates with hybrid search, rerank with cross-encoder to get top 5-10.

> Filtering good results into the best results

In [24]:
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

In [ ]:
# Cross-encoder model - trained specifically for reranking
# ms-marco-MiniLM is fast and effective for most use cases
cross_encoder = HuggingFaceCrossEncoder(
    model_name="cross-encoder/ms-marco-MiniLM-L6-v2"
)

# Reranker wrapper
reranker = CrossEncoderReranker(
    model=cross_encoder,
    top_n=5  # Return only top 5 after reranking all candidates
)

# Compression retriever orchestrates: retrieve → rerank → return
compression_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=hybrid_retriever
)

print("Reranking pipeline ready")
print(f"Flow: Hybrid retrieval → Cross-encoder scoring → Top {reranker.top_n} results")

In [26]:
# Compare: Before and After Reranking
query = "What's happening with Bitcoin ETFs?"

print("=" * 70)
print("BEFORE RERANKING (Hybrid Retrieval Only)")
print("=" * 70)

hybrid_results = hybrid_retriever.invoke(query)
for i, doc in enumerate(hybrid_results[:5], 1):
    print(f"\n[{i}] {doc.page_content[:120]}...")

print("\n" + "=" * 70)
print("AFTER RERANKING (Cross-Encoder Refined)")
print("=" * 70)

reranked_results = compression_retriever.invoke(query)
for i, doc in enumerate(reranked_results, 1):
    print(f"\n[{i}] {doc.page_content[:120]}...")

print("\n" + "=" * 70)
print("Notice how reranking reordered results based on deeper relevance")

BEFORE RERANKING (Hybrid Retrieval Only)

[1] Happy Thursday! Bitcoin (BTC) is holding a tight range around $68,000 as macro forces squeeze liquidity, with analysts s...

[2] Happy Wednesday! Bitcoin pushed back above $73,000 this morning as spot ETF inflows pick up again, with analysts saying ...

[3] Happy Monday! Bitcoin (BTC) is trading back above $69,000 despite escalating Middle East tensions and fading rate-cut ho...

[4] Happy Wednesday! Bitcoin (BTC) rebounded to $69,000 this morning after briefly touching its year-to-date low yesterday, ...

[5] The move follows $171 million in net outflows from U.S. spot bitcoin ETFs on March 26 , alongside continued outflows fro...

AFTER RERANKING (Cross-Encoder Refined)

[1] This trend is also spotted in the continued outflows from U.S. crypto exchange-traded funds. Bitcoin ETFs recently poste...

[2] The move follows $171 million in net outflows from U.S. spot bitcoin ETFs on March 26 , alongside continued outflows fro...

[3] Meanwhile,

## Step 6: Time-Aware Ranking
Crypto news is highly time-sensitive.
So prioritize the recent information.

How? Apply a freshness score
- Newer articles → higher weight
- Older articles → gradually reduced importance

This ensures users get current insights, not outdated ones

In [71]:
import math
import logging
import pytz
from datetime import datetime
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class ScoredChunk:
    """Single retrieved chunk with all scores attached."""
    content: str
    metadata: dict
    cross_encoder_score: float
    freshness_score: float
    final_score: float

    # Convenience shortcuts into metadata
    @property
    def article_id(self) -> str:
        return self.metadata.get("article_id", "")

    @property
    def title(self) -> str:
        return self.metadata.get("title", "")

    @property
    def published_at(self) -> str:
        return self.metadata.get("published_at", "")

    @property
    def url(self) -> str:
        return self.metadata.get("url", "")

    @property
    def tags(self) -> str:
        return self.metadata.get("tags", "")

    @property
    def chunk_index(self) -> int:
        return self.metadata.get("chunk_index", 0)

    def to_dict(self) -> dict:
        """Serialise to dict — ready to pass into format_context()."""
        return {
            **self.metadata,
            "content": self.content,
            "score": self.final_score,
        }


# Step 1 — Retrieve
def retrieve_docs(query: str, compression_retriever) -> list:
    docs = compression_retriever.invoke(query)
    print(f"Retrieved {len(docs)} docs for query: {query[:80]!r}")
    return docs

# Step 2 — Cross-encoder scoring
def score_with_cross_encoder(query: str, docs: list, cross_encoder) -> list[float]:
    if not docs:
        return []

    pairs = [(query, doc.page_content) for doc in docs]
    scores = cross_encoder.score(pairs)
    print(f"Cross-encoder scored {len(scores)} pairs")
    return list(scores)

# Step 3 — Freshness scoring
def compute_freshness(published_str: Optional[str], now: datetime, lambda_: float = 0.1) -> float:
    """
    Exponential time-decay freshness score.

    Score = exp(-lambda * age_in_days)
    - lambda_=0.1 → half-life ~7 days  (good for daily crypto news)
    - lambda_=0.01 → half-life ~70 days (slower decay for research articles)

    Args:
        published_str: ISO-8601 date string from metadata, or None.
        now:           Current UTC datetime.
        lambda_:       Decay rate.

    Returns:
        Float in (0, 1]. Returns 1.0 if date is missing or unparseable.
    """
    if not published_str:
        return 1.0
    try:
        published = datetime.fromisoformat(published_str)
        # Make timezone-aware if naive
        if published.tzinfo is None:
            published = published.replace(tzinfo=pytz.UTC)
        age_days = max((now - published).days, 0)
        return math.exp(-lambda_ * age_days)
    except (ValueError, TypeError) as exc:
        print(f"Could not parse published_at={published_str!r}: {exc}")
        return 1.0

# Step 4 — Combine scores
def combine_scores(docs: list, cross_scores: list[float], lambda_: float = 0.1, now: Optional[datetime] = None) -> list[ScoredChunk]:
    """
    Merge cross-encoder score and freshness into a final score.

    final_score = cross_encoder_score * freshness

    Args:
        docs:         List of LangChain Document objects.
        cross_scores: Raw scores from score_with_cross_encoder().
        lambda_:      Freshness decay rate passed to compute_freshness().
        now:          UTC datetime reference (defaults to datetime.now(UTC)).

    Returns:
        List of ScoredChunk, unsorted.
    """
    if now is None:
        now = datetime.now(pytz.UTC)

    results = []
    for doc, ce_score in zip(docs, cross_scores):
        published_str = doc.metadata.get("published_at")
        freshness = compute_freshness(published_str, now, lambda_)
        final = float(ce_score) * freshness

        results.append(
            ScoredChunk(
                content=doc.page_content,
                metadata=doc.metadata,
                cross_encoder_score=float(ce_score),
                freshness_score=freshness,
                final_score=final,
            )
        )

    return results

# Step 5 — Sort + optional top-n
def rank_chunks(scored_chunks: list[ScoredChunk], top_n: Optional[int] = None) -> list[ScoredChunk]:
    """
    Sort chunks by final_score descending and optionally truncate.
    """
    ranked = sorted(scored_chunks, key=lambda c: c.final_score, reverse=True)
    if top_n is not None:
        ranked = ranked[:top_n]
    return ranked

In [75]:
def retrieve_and_score(query: str, compression_retriever, cross_encoder, lambda_: float = 0.1, top_n: Optional[int] = None, now: Optional[datetime] = None) -> list[ScoredChunk]:
    if now is None:
        now = datetime.now(pytz.UTC)

    docs        = retrieve_docs(query, compression_retriever)
    ce_scores   = score_with_cross_encoder(query, docs, cross_encoder)
    scored      = combine_scores(docs, ce_scores, lambda_=lambda_, now=now)
    ranked      = rank_chunks(scored, top_n=top_n)

    print(
        f"retrieve_and_score | query={query[:60]} | docs={len(docs)} | returned={len(ranked)} | top_score={ranked[0].final_score if ranked else 0.0}.4f"
    )
    return ranked

In [ ]:
cross_encoder = HuggingFaceCrossEncoder(
    model_name="cross-encoder/ms-marco-MiniLM-L6-v2"
)

reranker = CrossEncoderReranker(
    model=cross_encoder,
    top_n=30
)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=hybrid_retriever
)

In [82]:
results = retrieve_and_score(
    query="Where does AlphaTON invested and how much?",
    compression_retriever=compression_retriever,
    cross_encoder=cross_encoder,
    top_n=5,
    lambda_=0.1,
)

# Inspect
for chunk in results:
  print(f"{chunk.final_score:.4f} | {chunk.published_at} | {chunk.content[:90]}")

Retrieved 8 docs for query: 'Where does AlphaTON invested and how much?'
Cross-encoder scored 8 pairs
retrieve_and_score | query=Where does AlphaTON invested and how much? | docs=8 | returned=5 | top_score=4.513114137338432.4f
4.5131 | 2026-04-09T18:23:40Z | AlphaTON Capital is looking to invest $43 million in its AI infrastructure operations, fin
3.7885 | 2026-04-09T18:23:40Z | Title: AlphaTON Capital looks to raise $43 million to bolster Telegram's Cocoon AI infrast
3.0702 | 2026-04-09T18:23:40Z | With the financing, AlphaTON, known for its support of the TON blockchain ecosystem and TO
0.4259 | 2026-04-09T18:23:40Z | AlphaTON was formed through a major pivot and rebranding of the existing public company fo
0.0168 | 2026-02-18T15:52:50Z | AlphaTON Capital Corp. (NASDAQ: ATON), the TON token-focused treasury and investment firm,


## Step 7: Answer Generation (LLM)

Now we have the best context.
We pass it to an LLM with a prompt.

What the LLM does:
- Reads retrieved chunks
- Synthesizes information
- Generates a coherent answer

Important constraint:
The model is instructed to use only provided context and avoid hallucinations

**Stack:** ChatGroq + LangChain + Pydantic structured output  

In [ ]:
import os
from datetime import datetime, timezone
from typing import Literal, Optional

from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

groq_api_key = os.getenv("GROQ_API_KEY","")

In [106]:
class SourceArticle(BaseModel):
    article_id: str
    title: str
    url: str
    published_at: str
    tags: list[str] = Field(default_factory=list)
    snippet: str = Field(description="Excerpt that directly supports the answer")
    relevance_score: float = Field(ge=0.0, le=1.0)
    chunk_index: int


class PriceSignal(BaseModel):
    asset: str                                             # "BTC", "ETH"
    direction: Literal["up", "down", "sideways", "unknown"]
    mentioned_price: Optional[str] = None                 # "$94,200" as-is


class RAGResponse(BaseModel):
    answer: str = Field(description="Factual answer grounded only in the articles")
    sources: list[SourceArticle]
    confidence: float = Field(ge=0.0, le=1.0)

    # Crypto-specific
    assets_mentioned: list[str] = Field(default_factory=list)
    price_signals: list[PriceSignal] = Field(default_factory=list)
    sentiment: Optional[Literal["bullish", "bearish", "neutral", "mixed"]] = None
    time_sensitivity: Literal["breaking", "recent", "historical"]

    # UX helpers
    needs_clarification: bool = False
    clarification_question: Optional[str] = None
    data_freshness_warning: Optional[str] = None

print("Schema defined ✓")

Schema defined ✓


In [86]:
SYSTEM = """\
You are a crypto news analyst. Answer questions using ONLY the articles below.

Rules:
- Cite every claim with the article_id of the source.
- Extract ticker symbols (BTC, ETH, SOL, ...) even when written as full names.
- If articles contradict each other, mention both sides and lower confidence.
- For price questions, always note how recent the data is.
- time_sensitivity: 'breaking' = <6 h, 'recent' = <7 days, 'historical' = older.
- Never speculate beyond the articles.

Current UTC time: {current_datetime}

--- ARTICLES ---
{context}
--- END ---
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    ("human", "{question}"),
])

print("Prompt template defined ✓")

Prompt template defined ✓


In [88]:
# llama-3.3-70b-versatile is the best Groq model for structured output
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=groq_api_key
)

structured_llm = llm.with_structured_output(RAGResponse)
chain = prompt | structured_llm

print("Chain ready ✓")

Chain ready ✓


In [101]:
# Context formatter + staleness guard
from dateutil import parser as dateparser

PRICE_KEYWORDS = {"price", "trading", "ath", "low", "high", "$", "usd", "pump", "dump", "rally"}
SEP = "\n" + "=" * 50 + "\n"


def format_context(chunks: list[dict]) -> str:
    """Convert your ranked chunks into a prompt-ready context block."""
    parts = []
    for c in chunks:
      tags = c.get("tags", "")
      parts.append(
          f"[article_id: {c['article_id']}]]\n"
          f"Title     : {c['title']}\n"
          f"Published : {c['published_at']}\n"
          f"URL       : {c['url']}\n"
          f"Tags      : {tags}\n"
          f"Score     : {c.get('score', 0):.3f}\n\n"
          f"{c['content']}"
      )
    return SEP.join(parts)


def staleness_warning(chunks: list[dict], question: str) -> Optional[str]:
    """Warn if question looks price-related but freshest article is >24 h old."""
    if not any(kw in question.lower() for kw in PRICE_KEYWORDS):
        return None
    try:
        dates = [dateparser.parse(c["published_at"]) for c in chunks if c.get("published_at")]
        freshest = max(d.replace(tzinfo=timezone.utc) for d in dates if d)
        age_h = (datetime.now(timezone.utc) - freshest).total_seconds() / 3600
        if age_h > 24:
            return f"Most recent article is {age_h:.0f}h old — price data may be outdated."
    except Exception:
        pass
    return None


print("Helpers defined ✓")

Helpers defined ✓


In [110]:
def answer_query(question: str, ranked_chunks: list[dict]) -> RAGResponse:
    """Synchronous wrapper — convenient for notebook use."""
    context = format_context(ranked_chunks)
    warning = staleness_warning(ranked_chunks, question)

    q = question
    if warning:
        q += f"\n\n[System note: {warning}]"

    print(q)
    return chain.invoke({
        "context": context,
        "question": q,
        "current_datetime": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC"),
    })

print("answer_query() ready ✓")

answer_query() ready ✓


In [112]:
query = "What is driving the BTC rally and how is ETH performing?"
results = retrieve_and_score(
    query=query,
    compression_retriever=compression_retriever,
    cross_encoder=cross_encoder,
    top_n=5,           # optional — was missing before
    lambda_=0.1,       # same decay rate you had
)

# Inspect
for chunk in results:
    print(f"{chunk.final_score:.4f} | {chunk.published_at} | {chunk.content[:90]}")

# Feed into your LLM layer
ranked_dicts = [c.to_dict() for c in results]

Retrieved 8 docs for query: 'What is driving the BTC rally and how is ETH performing?'
Cross-encoder scored 8 pairs
retrieve_and_score | query=What is driving the BTC rally and how is ETH performing? | docs=8 | returned=5 | top_score=3.5969046861229637.4f
3.5969 | 2026-04-09T20:11:49Z | Bitcoin (BTC) and ether (ETH) prices have moved higher, and the rally is driven by new lon
-0.0295 | 2026-03-06T12:46:45Z | Bitcoin’s (BTC) recent rally was powered by stronger spot demand and more than $1.1 billio
-0.0423 | 2026-03-06T12:46:45Z | Content: Bitcoin’s (BTC) recent rally was powered by stronger spot demand and more than $1
-0.0610 | 2026-02-18T17:31:45Z | Episode 7 of Layer One, hosted by The Block's Kelvin Sparks and Hypha founder Steven Gates
-0.1040 | 2026-03-05T21:04:41Z | Bitcoin’s rebound above $73,000 earlier Thursday may appear encouraging, but onchain data 


In [113]:
response = answer_query(
    question=query,
    ranked_chunks=ranked_dicts,
)

print("=" * 60)
print("ANSWER")
print("=" * 60)
print(response.answer)

print(f"\nConfidence      : {response.confidence:.0%}")
print(f"Time sensitivity: {response.time_sensitivity}")
print(f"Assets mentioned: {response.assets_mentioned}")

# if response.price_signals:
#     print("\nPrice signals:")
#     for sig in response.price_signals:
#         print(f"  {sig.asset}: {sig.direction} {sig.mentioned_price or ''}")

# print("\nSources:")
# for src in response.sources:
#     print(f"  [{src.article_id}] {src.published_at[:10]}  {src.title}")
#     print(f"   snippet: {src.snippet[:90]}...")

# if response.data_freshness_warning:
#     print(f"\n⚠ {response.data_freshness_warning}")

# if response.needs_clarification:
#     print(f"\nNeeds clarification: {response.clarification_question}")

What is driving the BTC rally and how is ETH performing?

[System note: Most recent article is 47h old — price data may be outdated.]
ANSWER
The BTC rally is driven by new long positions in perpetual futures, according to CryptoQuant (article_id: 396943). ETH is also performing well, with a 6% increase in price within 24 hours of the U.S.-Iran ceasefire announcement (article_id: 396943).

Confidence      : 80%
Time sensitivity: recent
Assets mentioned: ['BTC', 'ETH']
